# Adversarial LLM-Based Tabular Data Generator for Imbalanced Datasets
### Framework: QLoRA (4-bit BitsAndBytes) + Dual Evaluator Judges + Iterative DPO Fine-Tuning (TRL)

This notebook implements an **Adversarial & Preference-Guided LLM Tabular Generator** for minority-class oversampling, based on the following pipeline architecture:

```
[ Kaggle / Tabular Dataset ]
             │
             ▼
   [ Train / Test Split ]
      │               │
      │               ▼
      │        [ Test Evaluation ]
      │               ▲
      ├───► [ Model 1: Baseline RF ]
      │               ▲
      ▼               │ (Evaluates & acts as Judge B)
 [ Training Data ] ───┘
      │
      ▼
 ╔══════════════════════════════════════════════════════════════════════════╗
 ║                  BLACK BOX GENERATOR & DPO FEEDBACK LOOP                 ║
 ║                                                                          ║
 ║     ┌────────────────┐      Candidate Points     ┌─────────────────┐     ║
 ║     │ Local LLM      │ ────────────────────────► │ Judge A:        │     ║
 ║     │ (Qwen2.5-1.5B) │                           │ Realism Filter  │     ║
 ║     └───────▲────────┘                           │ (IsolationForest│     ║
 ║             │                                    └───────┬─────────┘     ║
 ║             │                                     Passes │    Fails      ║
 ║             │                                            ▼      │        ║
 ║             │                                    ┌─────────────────┐     ║
 ║             │                                    │ Judge B:        │     ║
 ║             │                                    │ Boundary Filter │     ║
 ║             │                                    │ (Baseline RF)   │     ║
 ║             │                                    └───────┬─────────┘     ║
 ║             │                                     Passes │    Fails      ║
 ║             │                                            ▼      │        ║
 ║             │                                      [Chosen]  [Rejected]  ║
 ║             │                                            │      │        ║
 ║             │                                            ▼      ▼        ║
 ║             │  DPO Updates (trl.DPOTrainer)        ┌─────────────────┐   ║
 ║             └───────────────────────────────────── │ Feedback Bucket │   ║
 ║                                                    └─────────────────┘   ║
 ╚══════════════════════════════════════════════════════════════════════════╝
      │
      ▼
 [ Mixed Data: Real Train + Valid Synthesized Minority ]
      │
      ▼
 [ Model 2: Augmented RF ] ───► [ Test Evaluation & Comparison ]
```




## 1. Setup & Environment Verification
Ensure CUDA is enabled and all necessary packages (`torch`, `transformers`, `peft`, `bitsandbytes`, `trl`, `datasets`, `kagglehub`, `scikit-learn`, `pandas`, `numpy`, `tabulate`) are imported.


In [14]:
import os
import gc
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import DPOTrainer, DPOConfig

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import (
    classification_report,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.datasets import make_classification
import kagglehub

warnings.filterwarnings("ignore")

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")



# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Windows / Non-Triton Compatibility Patch for TRL DPOTrainer
# In HuggingFace TRL 1.15+, fused_lm_head requires Triton kernels.
# On Windows, Triton is absent, making _ChunkedLogProbFunction None.
# We supply a high-efficiency pure-PyTorch autograd implementation with full metrics:
# ---------------------------------------------------------------------------
import trl.trainer.utils

class PureTorchChunkedLogProb(torch.autograd.Function):
    @staticmethod
    def forward(ctx, hidden, weight, bias, targets, temperature=1.0, chunk_size=32768, final_logit_softcapping=None, logit_scale=1.0, outputs=('log_probs',)):
        if hidden.numel() == 0:
            ctx.is_empty = True
            empty = hidden.new_empty(0)
            return empty, empty, empty, empty, empty.bool()
        ctx.is_empty = False
        
        with torch.amp.autocast(device_type=hidden.device.type, enabled=False):
            h_f = hidden.float()
            w_f = weight.float()
            logits = torch.matmul(h_f, w_f.t())
            if bias is not None:
                logits = logits + bias.float()
            logits = logits * (logit_scale / temperature)
            if final_logit_softcapping is not None:
                logits = final_logit_softcapping * torch.tanh(logits / final_logit_softcapping)
            log_probs = torch.log_softmax(logits, dim=-1)
            target_logprobs = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
            probs = torch.softmax(logits, dim=-1)
            
            entropy = -(probs * log_probs).sum(dim=-1)
            log_sum_sq = torch.log((probs ** 2).sum(dim=-1) + 1e-12)
            mean_logits = logits.mean(dim=-1)
            is_top1 = (logits.argmax(dim=-1) == targets)  # keep bool: TRL uses bitwise & with a bool mask

        ctx.save_for_backward(hidden, weight, targets, probs)
        ctx.temperature = temperature
        ctx.logit_scale = logit_scale
        ctx.has_bias = bias is not None
        return target_logprobs, entropy, log_sum_sq, mean_logits, is_top1

    @staticmethod
    def backward(ctx, grad_logprobs, *unused):
        if getattr(ctx, 'is_empty', False):
            return None, None, None, None, None, None, None, None, None
        hidden, weight, targets, probs = ctx.saved_tensors
        temperature = ctx.temperature
        logit_scale = ctx.logit_scale
        grad_logprobs_f = grad_logprobs.float().unsqueeze(-1)
        grad_logits = -probs * grad_logprobs_f
        grad_logits.scatter_add_(-1, targets.unsqueeze(-1), grad_logprobs_f)
        grad_logits = grad_logits * (logit_scale / temperature)
        grad_hidden = torch.matmul(grad_logits, weight.float()).to(hidden.dtype)
        grad_weight = torch.matmul(grad_logits.t(), hidden.float()).to(weight.dtype) if ctx.needs_input_grad[1] else None
        grad_bias = grad_logits.sum(dim=0).to(weight.dtype) if (ctx.has_bias and ctx.needs_input_grad[2]) else None
        return grad_hidden, grad_weight, grad_bias, None, None, None, None, None, None

trl.trainer.utils._ChunkedLogProbFunction = PureTorchChunkedLogProb
print('TRL Windows / CUDA compatibility patch registered with full metric support.')





Using device: cuda
Device Name: NVIDIA GeForce RTX 5050 Laptop GPU
Allocated VRAM: 1104.08 MB
TRL Windows / CUDA compatibility patch registered with full metric support.


## 2. Dataset Ingestion (Direct Kaggle Import or Synthetic)
We support direct 1-click downloading of public Kaggle datasets (e.g. Credit Card Customers Churn dataset `sakshigoyal7/credit-card-customers` or Credit Card Fraud `mlg-ulb/creditcardfraud`) via `kagglehub.dataset_download()`.
If offline or desired, a synthetic imbalanced classification dataset is also available with a single toggle.


In [15]:
# Configuration options
USE_KAGGLE = True  # Set to False to use synthetic make_classification
KAGGLE_DATASET_SLUG = "sakshigoyal7/credit-card-customers"  # Highly imbalanced churn classification

def load_imbalanced_data(use_kaggle=True, random_state=42):
    if use_kaggle:
        print(f"Fetching '{KAGGLE_DATASET_SLUG}' directly from Kaggle via kagglehub...")
        try:
            dataset_path = kagglehub.dataset_download(KAGGLE_DATASET_SLUG)
            print(f"Downloaded to: {dataset_path}")
            
            # Find the main CSV file
            csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
            csv_path = os.path.join(dataset_path, csv_files[0])
            print(f"Loading CSV: {csv_path}")
            df = pd.read_csv(csv_path)
            
            # For BankChurners dataset: predict 'Attrition_Flag' (Attrited Customer = 1, Existing Customer = 0)
            if "Attrition_Flag" in df.columns:
                target_col = "Attrition_Flag"
                df[target_col] = (df[target_col] == "Attrited Customer").astype(int)
                # Drop leakage / ID columns
                cols_to_drop = [c for c in df.columns if "CLIENTNUM" in c or "Naive_Bayes" in c or c == target_col]
                # Select key numerical columns for clear LLM serialization
                num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
                num_cols = [c for c in num_cols if c not in cols_to_drop][:8]  # Pick 8 most informative numerical features
                
                X = df[num_cols].copy()
                y = df[target_col].copy()
            else:
                # Generic binary target detection
                target_candidates = [c for c in df.columns if df[c].nunique() == 2]
                target_col = target_candidates[-1]
                X = df.select_dtypes(include=[np.number]).drop(columns=[target_col], errors='ignore').iloc[:, :8]
                y = (df[target_col] == df[target_col].unique()[1]).astype(int)
                
            print(f"Kaggle Dataset loaded successfully: {X.shape[0]} rows, {X.shape[1]} features.")
        except Exception as e:
            print(f"Failed to load Kaggle dataset ({e}). Falling back to synthetic make_classification.")
            use_kaggle = False

    if not use_kaggle:
        print("Generating synthetic imbalanced binary classification dataset...")
        feature_names = [f"feat_{i}" for i in range(8)]
        X_raw, y_raw = make_classification(
            n_samples=3000,
            n_features=8,
            n_informative=6,
            n_redundant=2,
            weights=[0.95, 0.05],  # 95% Majority (Class 0), 5% Minority (Class 1)
            flip_y=0.01,
            random_state=random_state,
        )
        X = pd.DataFrame(X_raw, columns=feature_names)
        y = pd.Series(y_raw, name="target")

    # Round numerical features to 2 decimals for clean token serialization
    X = X.round(2)
    return X, y

X, y = load_imbalanced_data(use_kaggle=USE_KAGGLE)
feature_names = list(X.columns)

# Train / Test Split (Stratified to maintain class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Print Class Distribution
print("\n--- Class Distribution in Training Set ---")
train_counts = y_train.value_counts()
print(f"Majority Class 0: {train_counts[0]} ({train_counts[0] / len(y_train):.1%})")
print(f"Minority Class 1: {train_counts[1]} ({train_counts[1] / len(y_train):.1%})")
print(f"Imbalance Ratio:  1 : {train_counts[0] / train_counts[1]:.1f}")




Fetching 'sakshigoyal7/credit-card-customers' directly from Kaggle via kagglehub...
Downloaded to: C:\Users\Shravya Patel\.cache\kagglehub\datasets\sakshigoyal7\credit-card-customers\versions\1
Loading CSV: C:\Users\Shravya Patel\.cache\kagglehub\datasets\sakshigoyal7\credit-card-customers\versions\1\BankChurners.csv
Kaggle Dataset loaded successfully: 10127 rows, 8 features.

--- Class Distribution in Training Set ---
Majority Class 0: 6375 (83.9%)
Minority Class 1: 1220 (16.1%)
Imbalance Ratio:  1 : 5.2


## 3. Baseline Model (Model 1)
We train a `RandomForestClassifier` on the raw, imbalanced training data to establish our baseline performance and act as **Judge B** (Class Boundary Evaluator).


In [16]:
# Train Baseline RandomForestClassifier
baseline_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
baseline_rf.fit(X_train, y_train)

# Evaluate on Test Set
y_pred_baseline = baseline_rf.predict(X_test)
y_prob_baseline = baseline_rf.predict_proba(X_test)[:, 1]

print("=== Baseline Model 1 Performance (Imbalanced Training Set) ===")
print(classification_report(y_test, y_pred_baseline, digits=4))
baseline_recall = recall_score(y_test, y_pred_baseline, pos_label=1)
baseline_f1 = f1_score(y_test, y_pred_baseline, pos_label=1)
baseline_precision = precision_score(y_test, y_pred_baseline, pos_label=1)
baseline_auc = roc_auc_score(y_test, y_prob_baseline)

print(f"Minority Class (1) Precision: {baseline_precision:.4f}")
print(f"Minority Class (1) Recall:    {baseline_recall:.4f}")
print(f"Minority Class (1) F1-Score:  {baseline_f1:.4f}")
print(f"ROC-AUC:                      {baseline_auc:.4f}")




=== Baseline Model 1 Performance (Imbalanced Training Set) ===
              precision    recall  f1-score   support

           0     0.8802    0.9816    0.9281      2125
           1     0.7593    0.3022    0.4323       407

    accuracy                         0.8724      2532
   macro avg     0.8197    0.6419    0.6802      2532
weighted avg     0.8607    0.8724    0.8484      2532

Minority Class (1) Precision: 0.7593
Minority Class (1) Recall:    0.3022
Minority Class (1) F1-Score:  0.4323
ROC-AUC:                      0.8540


## 4. Data Serialization Layer
To communicate tabular feature rows with the LLM:
1. `serialize_row(row)` converts a dictionary or pandas row into clean structured JSON format.
2. `deserialize_response(text)` extracts, parses, and validates numerical JSON objects generated by the LLM with fallback regex extraction and type casting.


In [17]:
def serialize_row(row) -> str:
    """Converts a tabular feature row into structured JSON string format."""
    if isinstance(row, pd.Series):
        d = row.to_dict()
    elif isinstance(row, dict):
        d = row
    else:
        d = dict(zip(feature_names, row))
    # Round numerical values to 2 decimal places
    clean_dict = {k: round(float(v), 2) for k, v in d.items() if k in feature_names}
    return json.dumps(clean_dict)

def deserialize_response(text: str) -> dict:
    """Parses raw LLM text into a validated numerical feature dictionary.
    Returns None if text cannot be deserialized or has invalid schema."""
    try:
        # Search for JSON object within text {...}
        json_match = re.search(r'\{[^{}]*\}', text)
        if json_match:
            parsed = json.loads(json_match.group(0))
            # Verify feature names and float conversion
            cleaned = {}
            for col in feature_names:
                if col in parsed:
                    cleaned[col] = float(parsed[col])
                else:
                    return None  # Missing essential column
            return cleaned
    except Exception:
        pass
    return None

# Test Serialization Layer
sample_minority_row = X_train[y_train == 1].iloc[0]
serialized_sample = serialize_row(sample_minority_row)
deserialized_sample = deserialize_response(serialized_sample)

print("Serialized Sample:")
print(serialized_sample)
print("\nDeserialized Verification:")
print(deserialized_sample)
assert deserialized_sample is not None, "Serialization validation failed!"




Serialized Sample:
{"Customer_Age": 44.0, "Dependent_count": 5.0, "Months_on_book": 37.0, "Total_Relationship_Count": 1.0, "Months_Inactive_12_mon": 2.0, "Contacts_Count_12_mon": 4.0, "Credit_Limit": 1491.0, "Total_Revolving_Bal": 0.0}

Deserialized Verification:
{'Customer_Age': 44.0, 'Dependent_count': 5.0, 'Months_on_book': 37.0, 'Total_Relationship_Count': 1.0, 'Months_Inactive_12_mon': 2.0, 'Contacts_Count_12_mon': 4.0, 'Credit_Limit': 1491.0, 'Total_Revolving_Bal': 0.0}


## 5. Evaluator Judges (Adversarial Quality Filters)
Following the architecture:
- **Judge A (Realism Filter)**: `IsolationForest` trained strictly on real minority class training data ($y=1$). Identifies realistic feature correlations and flags impossible anomalies.
- **Judge B (Class Boundary Filter)**: The baseline `RandomForestClassifier` trained in Step 3. Checks whether the generated point convincingly resides inside the minority class boundary (e.g. $P(	ext{class}=1) \ge 0.65$).


In [18]:
# Filter minority class training data
X_minority_train = X_train[y_train == 1].copy()

# Judge A: IsolationForest on real minority samples
judge_a_realism = IsolationForest(
    n_estimators=100,
    contamination=0.08,  # Outlier tolerance
    random_state=42
)
judge_a_realism.fit(X_minority_train)

# Judge B: Baseline Random Forest model
judge_b_boundary = baseline_rf

def evaluate_candidate(point_dict, boundary_threshold=0.55):  # Relaxed: 0.55 to pass more points
    """
    Evaluates candidate point through Judge A and Judge B.
    Returns:
        is_valid (bool): True if passes both judges.
        decision (dict): Detailed judgment scores and flags.
    """
    if point_dict is None:
        return False, {"reason": "parse_failure", "realism": False, "p_minority": 0.0}
    
    # Format point as 1-row DataFrame with identical columns
    pt_df = pd.DataFrame([point_dict])[feature_names]
    
    # 1. Judge A: Realism test (IsolationForest returns 1 for inlier, -1 for outlier)
    iso_pred = judge_a_realism.predict(pt_df)[0]
    passes_realism = (iso_pred == 1)
    
    # 2. Judge B: Boundary test (Probability of Class 1)
    p_minority = float(judge_b_boundary.predict_proba(pt_df)[0, 1])
    passes_boundary = (p_minority >= boundary_threshold)
    
    is_valid = passes_realism and passes_boundary
    reason = "valid" if is_valid else ("failed_realism" if not passes_realism else "failed_boundary")
    
    return is_valid, {
        "reason": reason,
        "passes_realism": passes_realism,
        "passes_boundary": passes_boundary,
        "p_minority": p_minority
    }

print("Judges initialized successfully.")
test_validity, test_info = evaluate_candidate(sample_minority_row.to_dict())
print(f"Validation on real minority row: Passes = {test_validity}, Info = {test_info}")





Judges initialized successfully.
Validation on real minority row: Passes = True, Info = {'reason': 'valid', 'passes_realism': np.True_, 'passes_boundary': True, 'p_minority': 0.840139747556248}


## 6. Base Model Setup with 4-Bit QLoRA
We load a lightweight, high-performance instruction model (`Qwen/Qwen2.5-1.5B-Instruct`) in 4-bit NormalFloat precision via `BitsAndBytesConfig`.
We then attach Low-Rank Adapters (LoRA) on `q_proj` and `v_proj` using `peft.LoraConfig`.


In [19]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load 4-bit Base Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True,
)

# Attach LoRA Adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()




Loading weights: 100%|██████████| 338/338 [00:05<00:00, 65.35it/s]


trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


## 7. Prompt Strategy & Minority Generation Function
We construct a targeted generation prompt containing in-context seed minority rows to guide the LLM's feature distribution and formatting.


In [20]:
def create_generation_prompt(n_seed_examples=3):
    """Builds a structured prompt with seed real minority examples."""
    seed_samples = X_minority_train.sample(min(n_seed_examples, len(X_minority_train)), random_state=np.random.randint(1000))
    seed_str = "\n".join([f"Example {i+1}: " + serialize_row(row) for i, (_, row) in enumerate(seed_samples.iterrows())])
    
    prompt = (
        "You are an expert tabular generative AI. Your task is to generate one new, highly realistic "
        "numerical data record belonging strictly to Class 1 (Minority class).\n"
        "Follow the exact feature distribution and valid schema shown in these real Class 1 examples:\n"
        f"{seed_str}\n\n"
        "Generate exactly ONE new Class 1 row as a valid JSON dictionary containing all feature keys: "
        f"{feature_names}. Do NOT output explanation, markdown ticks, or text other than the JSON object."
    )
    return prompt

def generate_candidates(model, tokenizer, num_candidates=10, temperature=0.85):
    """Generates candidate textual tabular rows using the current LLM state."""
    model.eval()
    candidates = []
    
    # Fix: One consistent prompt per round for valid DPO pairs
    prompt_text = create_generation_prompt(n_seed_examples=2)
    messages = [
        {"role": "system", "content": "You are a tabular data generator. Output only valid JSON format."},
        {"role": "user", "content": prompt_text}
    ]
    
    for _ in range(num_candidates):
        
        # Apply chat template
        prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_formatted, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=160,
                temperature=temperature,
                top_p=0.92,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        # Extract response text
        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        candidates.append((prompt_formatted, response_text))
        
    return candidates

print("Generation pipeline ready.")





Generation pipeline ready.


## 8. Iterative Adversarial Generation & DPO Fine-Tuning Loop
### Execution Workflow (3-5 Rounds):
1. **Generate**: LLM generates candidate textual points for Class 1.
2. **Evaluate**: Parse response into numerical vectors, evaluate via Judge A (Realism) and Judge B (Boundary).
3. **Bucket Routing**:
   - Points passing both judges $\to$ saved to `accumulated_synthetic_data` and marked as `chosen`.
   - Points failing either judge $\to$ marked as `rejected`.
4. **Preference Triplets**: Construct `(prompt, chosen, rejected)` micro-dataset.
5. **DPO Training**: Fine-tune LoRA adapters via `trl.DPOTrainer` for 1-2 epochs to steer the LLM away from rejected boundaries toward approved distributions.


In [22]:
from trl import SFTTrainer, DPOConfig, DPOTrainer
from transformers import TrainingArguments

NUM_ROUNDS = 4
CANDIDATES_PER_ROUND = 25
EPOCHS_PER_ROUND = 2
FINAL_CANDIDATES = 100

def reset_lora_adapter(model):
    print("Resetting LoRA weights for fresh experiment...")
    if hasattr(model, 'peft_config'):
        if 'default' in model.peft_config:
            model.delete_adapter('default')
        if 'ref' in model.peft_config:
            model.delete_adapter('ref')
        model.add_adapter('default', lora_config)
    return model

def run_experiment(method="DPO"):
    print(f"\n{'='*50}")
    print(f"STARTING EXPERIMENT: {method}")
    print(f"{'='*50}\n")
    
    global model
    model = reset_lora_adapter(model)
    
    accumulated_synthetic_data = []
    
    for round_num in range(1, NUM_ROUNDS + 1):
        print("\n--- {method} ROUND {round_num}/{NUM_ROUNDS} ---")
        
        # 1. Generate Candidates (One consistent prompt per round)
        prompt_text = create_generation_prompt(n_seed_examples=2)
        messages = [
            {"role": "system", "content": "You are a tabular data generator. Output only valid JSON format."},
            {"role": "user", "content": prompt_text}
        ]
        
        raw_candidates = []
        for _ in range(CANDIDATES_PER_ROUND):
            prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(prompt_formatted, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=160, temperature=0.85, top_p=0.92,
                    do_sample=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
                )
            generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
            response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
            raw_candidates.append((prompt_formatted, response_text))
            
        round_chosen = []
        round_rejected = []
        
        # 2. Evaluate
        for prompt_str, resp_text in raw_candidates:
            parsed_dict = deserialize_response(resp_text)
            is_valid, eval_meta = evaluate_candidate(parsed_dict)
            if is_valid:
                round_chosen.append((prompt_str, resp_text, parsed_dict))
                accumulated_synthetic_data.append(parsed_dict)
            else:
                round_rejected.append((prompt_str, resp_text, eval_meta['reason']))
                
        yield_pct = (len(round_chosen) / CANDIDATES_PER_ROUND) * 100
        print(f"-> Acceptance Yield: {len(round_chosen)} Valid, {len(round_rejected)} Invalid ({yield_pct:.1f}%)")
        
        # 3. Form Dataset & Train based on method
        if method == "SFT":
            if len(round_chosen) == 0:
                print("No valid points to train on this round. Skipping SFT.")
                continue
                
            sft_data = [{"text": c_prompt + c_resp} for c_prompt, c_resp, _ in round_chosen]
            train_dataset = Dataset.from_pandas(pd.DataFrame(sft_data))
            
            args = TrainingArguments(
                output_dir=f"./tmp_sft_{round_num}",
                learning_rate=3e-5,
                per_device_train_batch_size=2,
                gradient_accumulation_steps=2,
                num_train_epochs=EPOCHS_PER_ROUND,
                logging_steps=1,
                save_strategy="no",
                report_to="none",
                max_steps=-1
            )
            trainer = SFTTrainer(
                model=model,
                args=args,
                train_dataset=train_dataset,
                dataset_text_field="text",
                max_seq_length=512,
            )
            
        elif method == "DPO":
            if len(round_chosen) == 0 or len(round_rejected) == 0:
                print("Need both chosen and rejected for preference learning. Skipping round.")
                continue
                
            pref_data = []
            for i in range(max(len(round_chosen), len(round_rejected))):
                c_prompt, c_resp, _ = round_chosen[i % len(round_chosen)]
                r_prompt, r_resp, _ = round_rejected[i % len(round_rejected)]
                pref_data.append({"prompt": c_prompt, "chosen": c_resp, "rejected": r_resp})
                
            train_dataset = Dataset.from_pandas(pd.DataFrame(pref_data))
            
            if method == "DPO":
                if hasattr(model, 'peft_config') and 'ref' in model.peft_config:
                    model.delete_adapter('ref')
                    model.set_adapter('default')
                    
                args = DPOConfig(
                    output_dir=f"./tmp_dpo_{round_num}",
                    learning_rate=3e-5,
                    per_device_train_batch_size=2,
                    gradient_accumulation_steps=2,
                    num_train_epochs=EPOCHS_PER_ROUND,
                    beta=0.1,
                    logging_steps=1,
                    save_strategy="no",
                    report_to="none",
                    max_length=512,
                    remove_unused_columns=False,
                )
                trainer = DPOTrainer(
                    model=model,
                    ref_model=None,
                    args=args,
                    train_dataset=train_dataset,
                    processing_class=tokenizer,
                )
            else: # ORPO
                args = ORPOConfig(
                    output_dir=f"./tmp_orpo_{round_num}",
                    learning_rate=3e-5,
                    per_device_train_batch_size=2,
                    gradient_accumulation_steps=2,
                    num_train_epochs=EPOCHS_PER_ROUND,
                    beta=0.1, # ORPO odds ratio weight
                    logging_steps=1,
                    save_strategy="no",
                    report_to="none",
                    max_length=512,
                    remove_unused_columns=False,
                )
                trainer = ORPOTrainer(
                    model=model,
                    args=args,
                    train_dataset=train_dataset,
                    processing_class=tokenizer,
                )
                
        print(f"[{method}] Fine-tuning...")
        trainer.train()
        del trainer
        torch.cuda.empty_cache()
        import gc; gc.collect()
        
    # Final Generation Run
    print("\n--- {method} FINAL GENERATION ---")
    prompt_text = create_generation_prompt(n_seed_examples=2)
    messages = [
        {"role": "system", "content": "You are a tabular data generator. Output only valid JSON format."},
        {"role": "user", "content": prompt_text}
    ]
    prompt_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    final_valid_points = []
    for _ in range(FINAL_CANDIDATES):
        inputs = tokenizer(prompt_formatted, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=160, temperature=0.7, top_p=0.92,
                do_sample=True, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            )
        generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        resp_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        
        parsed_dict = deserialize_response(resp_text)
        is_valid, _ = evaluate_candidate(parsed_dict)
        if is_valid:
            final_valid_points.append(parsed_dict)
            
    print(f"{method} Final Yield: {len(final_valid_points)}/{FINAL_CANDIDATES} ({(len(final_valid_points)/FINAL_CANDIDATES)*100:.1f}%)")
    
    all_data = accumulated_synthetic_data + final_valid_points
    return pd.DataFrame(all_data)[feature_names]

# Run all 3 methods!
results = {}
for method in ["SFT", "DPO"]:
    results[method] = run_experiment(method)




STARTING EXPERIMENT: SFT

Resetting LoRA weights for fresh experiment...

--- {method} ROUND {round_num}/{NUM_ROUNDS} ---
-> Acceptance Yield: 8 Valid, 17 Invalid (32.0%)


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'

## 10. Multi-Method Evaluation
We now have synthetic datasets generated by SFT, ORPO, and DPO aligned models. Let's augment the original dataset with each, train a classifier, and compare their downstream performance.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

base_preds = baseline_rf.predict(X_test)
base_probs = baseline_rf.predict_proba(X_test)[:, 1]
print('=== BASELINE (No Augmentation) ===')
print(f'ROC AUC Score: {roc_auc_score(y_test, base_probs):.4f}\n')

for method, synth_df in results.items():
    print(f'\n{'='*20} {method} AUGMENTATION {'='*20}')
    print(f'Synthetic Points Added: {len(synth_df)}')
    if len(synth_df) == 0:
        print('No valid data generated.')
        continue
        
    X_train_aug = pd.concat([X_train, synth_df], ignore_index=True)
    y_train_aug = pd.concat([y_train, pd.Series([1]*len(synth_df))], ignore_index=True)
    
    aug_rf = RandomForestClassifier(n_estimators=100, random_state=42)
    aug_rf.fit(X_train_aug, y_train_aug)
    
    aug_preds = aug_rf.predict(X_test)
    aug_probs = aug_rf.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, aug_preds))
    print(f'ROC AUC Score: {roc_auc_score(y_test, aug_probs):.4f}')

